# Assignment 2. Sensitivity Analysis: Which Inputs Matter?

**Course:** EPA141A Model Based Decision Making — Delft University of Technology  
**Model:** JUSTICE

---

## Learning Outcomes

After completing this assignment you will be able to:
1. Explain the purpose of sensitivity analysis and when to apply it.
2. Compute **Extra-Trees feature importance** using the EMA Workbench `feature_scoring` module.
3. Run a **Morris elementary effects** analysis with SALib and interpret **μ\*** and **σ**.
4. Explain why Sobol variance decomposition requires smooth, quasi-monotonic model responses,
   and identify the diagnostic symptoms (S₁ > 1, S₁ < 0) that indicate it has been misapplied.
5. Compare sensitivity rankings under **no abatement** vs. **moderate abatement** and explain
   what policy-conditional sensitivity means for decision-making.


---

## Background

Assignment 1 showed that four uncertain parameters produce a wide spread in all four outcomes.
Sensitivity analysis (SA) answers the follow-up question: **which inputs are responsible for that spread?**

Knowing this helps to:
- Prioritise where to invest in further research (reduce the most influential uncertainties first).
- Identify parameters that can safely be fixed without losing model fidelity.
- Understand whether the dominant driver of variability is physical (`ecs_ensemble`) or normative (ρ, η, δ).

We use two complementary methods that together give a complete picture:

**Extra-Trees feature importance** (EMA Workbench `feature_scoring`)  
Fits an ensemble of extremely randomised regression trees on the LHS ensemble from Step 2.
Reports how much each parameter reduces prediction variance across all trees. Fast, non-parametric,
and captures non-linear effects including `ecs_ensemble` (a discrete non-monotonic index).

**Morris elementary effects** (SALib `morris`)  
Perturbs each input one at a time from many random starting points and records the resulting
output change. Computes μ\* (mean absolute effect — importance) and σ (standard deviation —
non-linearity / interaction proxy). Morris handles non-monotonic and sign-changing model
responses reliably, making it the appropriate complement to ET for JUSTICE's welfare function.

### Why not Sobol for this model?
Sobol variance decomposition requires smooth, quasi-monotonic outputs. JUSTICE's `welfare`
changes sign at η ≈ 1.05 — from +856 at η = 0.9 to −551 at η = 1.1 — because the utility
curvature parameter changes the sign of the welfare aggregation. This makes the Saltelli
estimator unstable (S₁ > 1, S₁ < 0 are the diagnostic symptoms). Morris and ET are the
correct tools for this model.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from SALib.sample import morris as morris_sample
from SALib.analyze import morris as morris_analyze
from ema_workbench import (
    Model, RealParameter, ScalarOutcome,
    perform_experiments, ema_logging,
)
from ema_workbench.em_framework.evaluators import SequentialEvaluator
from ema_workbench.em_framework.samplers import Sample
from ema_workbench.analysis import feature_scoring

_NOTEBOOK_DIR = os.path.abspath('')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, '../JUSTICE-main'))
if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction

OBJECTIVES   = ['welfare', 'years_above_temperature_threshold',
                 'welfare_loss_damage', 'welfare_loss_abatement']
PARAMS       = ['rho', 'eta', 'delta', 'ecs_ensemble']
POLICY_NAMES = ['no_abatement', 'moderate_abatement']
palette      = {'no_abatement': 'steelblue', 'moderate_abatement': 'darkorange'}


## Step 1 — Define the model

The model wrapper is identical to Assignment 1: four uncertain parameters plus the `ecr_plateau` lever.

**Task 1.1** — Complete `justice_model` so it:
1. Hard-resets JUSTICE and instantiates it with the given `ecs_ensemble` index.
2. Sets ρ and η on both `model.economy` and `model.welfare_function`.
3. Scales the three damage coefficients by δ.
4. Runs with a uniform ECR equal to `ecr_plateau` across all regions and returns all four scalar outcomes.

**Task 1.2** — Run the smoke test below and confirm the four outcomes are positive and in plausible ranges.


In [ ]:
def justice_model(rho=0.015, eta=1.45, delta=1.0, ecs_ensemble=1, ecr_plateau=0.0):
    """
    EMA Workbench function model interface for JUSTICE.
    Adjust the ECR level via ecr_plateau (0.0 = no abatement, 0.4 = moderate).
    """
    JUSTICE.hard_reset()
    ensemble_idx = int(np.round(np.clip(ecs_ensemble, 1, 1001)))
    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=2, climate_ensembles=ensemble_idx,
        social_welfare_function=WelfareFunction.UTILITARIAN,
    )
    # TODO: set rho and eta on both model.economy and model.welfare_function
    # TODO: scale the three damage coefficients by delta
    # ...

    ecr = np.full((57, 286), ecr_plateau)
    datasets = model.run(ecr)

    welfare      = float(datasets['welfare'][0])
    yat          = float(np.sum(datasets['global_temperature'].mean(axis=1) > 2.0))
    wl_damage    = abs(float(datasets.get('welfare_loss_damage',    [0])[0]))
    wl_abatement = abs(float(datasets.get('welfare_loss_abatement', [0])[0]))

    return dict(
        welfare=welfare,
        years_above_temperature_threshold=yat,
        welfare_loss_damage=wl_damage,
        welfare_loss_abatement=wl_abatement,
    )


## Step 2 — EMA model setup and LHS ensemble

**Task 2.1** — Wrap `justice_model` in a `Model` object with the same four uncertain
parameters and four scalar outcomes as Assignment 1.

**Task 2.2** — Run 100 LHS scenarios under each policy using `SequentialEvaluator`. This
ensemble is used as input to the Extra-Trees analysis in Step 3.

**Task 2.3** — How does the `welfare_loss_abatement` distribution compare between the
two policies? Explain the difference.

---

*Note: The outcome distributions for both policies are fully explored in **Assignment 1**
(Steps 3–4). The key finding there was that `years_above_temperature_threshold` shows the
clearest policy separation (~15 year shift), while `welfare` distributions overlap almost
entirely — policy signal is overwhelmed by parametric uncertainty. SA in this assignment
explains **why** those distributions are wide and which specific inputs drive the spread.*


In [ ]:
em_model = Model('JUSTICE', function=justice_model)
em_model.uncertainties = [
    RealParameter('rho',          0.001,  0.030),
    RealParameter('eta',          0.5,    1.5),
    RealParameter('delta',        0.5,    2.0),
    RealParameter('ecs_ensemble', 1,   1001),
]
em_model.outcomes = [
    ScalarOutcome('welfare'),
    ScalarOutcome('years_above_temperature_threshold'),
    ScalarOutcome('welfare_loss_damage'),
    ScalarOutcome('welfare_loss_abatement'),
]

policies = [
    Sample('no_abatement',       ecr_plateau=0.0),
    Sample('moderate_abatement', ecr_plateau=0.4),
]

with SequentialEvaluator(em_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(
        scenarios=100, policies=policies)

df_results = pd.DataFrame(outcomes)
df_results['policy'] = experiments['policy'].values


## Step 3 — Extra-Trees feature importance

`feature_scoring.get_feature_scores_all(x, y)` fits an `ExtraTreesRegressor` for each
outcome on the LHS ensemble and returns a DataFrame of importances
(rows = parameters, columns = outcomes).

Running the analysis separately for each policy tests whether sensitivity is
**policy-conditional** — i.e., whether the same parameter matters equally under
both no abatement and moderate abatement.

**Task 3.1** — Compute Extra-Trees importances for each policy separately.

**Task 3.2** — Plot the importances as a 2×2 bar-chart grid for each policy (two grids total).

**Task 3.3** — Which parameter dominates `years_above_temperature_threshold`? Is this a
normative or physical uncertainty? What does this imply for climate policy?


In [ ]:
et_scores = {}
for pol in POLICY_NAMES:
    mask  = experiments['policy'] == pol
    x_pol = experiments[mask][PARAMS]
    y_pol = {obj: outcomes[obj][mask] for obj in OBJECTIVES}
    et_scores[pol] = feature_scoring.get_feature_scores_all(x_pol, y_pol)

# Plot 2×2 bar-chart grid per policy
for pol in POLICY_NAMES:
    scores = et_scores[pol]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(f'Extra-Trees feature importance — {pol}')
    for ax, obj in zip(axes.flat, OBJECTIVES):
        ax.bar(PARAMS, scores[obj], color=palette[pol])
        ax.axhline(0.25, color='gray', linestyle='--', linewidth=0.8)
        ax.set_title(obj.replace('years_above_temperature_threshold', 'yrs > 2°C'))
        ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()


## Step 4 — Policy-comparison sensitivity heatmap

A side-by-side heatmap (one panel per policy) puts both ET analyses on the same axes,
making it easy to see which parameters shift in importance between the two policies.
Importances are normalised per column so they sum to 1 within each outcome.

**Task 4.1** — Build the two normalised heatmaps and compare them.

**Task 4.2** — Which parameter gains the most importance under no abatement compared to
moderate abatement? Which loses importance? Explain both changes mechanistically.

**Task 4.3** — What does this imply for a decision-maker who needs to choose between the
two policies under deep uncertainty?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pol in zip(axes, POLICY_NAMES):
    scores = et_scores[pol]
    short  = scores.rename(index={
        'years_above_temperature_threshold': 'yrs > 2°C',
        'welfare_loss_damage':    'wl_damage',
        'welfare_loss_abatement': 'wl_abatement',
    })
    norm = short / short.sum()
    sns.heatmap(norm, annot=True, fmt='.2f', cmap='YlOrRd',
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title(f'ET importance (normalised) — {pol}')
plt.tight_layout()
plt.show()


## Step 5 — Morris elementary effects

The **Morris method** (elementary effects) is a global screening approach that perturbs one
input at a time by a finite step and computes the resulting elementary effect on the output.
For each input it summarises these effects with:
- **μ\*** — mean of absolute elementary effects (importance measure, always non-negative)
- **σ** — standard deviation (indicator of non-linearity and interactions)

### Why Morris instead of Sobol here?
Sobol variance decomposition requires smooth, quasi-monotonic outputs. JUSTICE's `welfare`
changes sign at η ≈ 1.05 — the utility curvature flips the sign of the welfare aggregation —
making the Saltelli estimator unstable (diagnostic: S₁ > 1 or S₁ < 0). Morris only relies
on local finite differences, so μ\* remains well-defined and interpretable for any model
structure. `ecs_ensemble` is excluded from Morris because it is a discrete index into the
FaIR ensemble; its influence is already captured by Extra-Trees in Step 3.

**Task 5.1** — Complete the Morris analysis below. Use `N_MORRIS = 50` trajectories
(→ 200 evaluations per policy). Fix `ecs_ensemble` at its median value (501).

**Task 5.2** — Plot μ\* and σ for each parameter and outcome (2×2 grid per policy).

**Task 5.3** — Compare Morris μ\* rankings to the Extra-Trees rankings from Step 3.
Where do they agree? Where do they differ, and why?


In [ ]:
N_MORRIS   = 50    # → 50 × (3+1) = 200 evaluations per policy
ECS_MEDIAN = 501   # fix ecs_ensemble at median of [1, 1001]

morris_problem = {
    'num_vars': 3,
    'names':    ['delta', 'eta', 'rho'],
    'bounds':   [[0.5, 2.0], [0.5, 1.5], [0.001, 0.030]],
}

morris_results = {}
for pol, ecr_val in [('no_abatement', 0.0), ('moderate_abatement', 0.4)]:
    param_samples = morris_sample.sample(morris_problem, N=N_MORRIS, num_levels=4)
    Y = {obj: np.zeros(len(param_samples)) for obj in OBJECTIVES}
    for j, (d, e, r) in enumerate(param_samples):
        res = justice_model(rho=r, eta=e, delta=d,
                            ecs_ensemble=ECS_MEDIAN, ecr_plateau=ecr_val)
        for obj in OBJECTIVES:
            Y[obj][j] = res[obj]
    morris_results[pol] = {
        obj: morris_analyze.analyze(morris_problem, param_samples, Y[obj])
        for obj in OBJECTIVES
    }

MORRIS_PARAMS = ['delta', 'eta', 'rho']
x = np.arange(len(MORRIS_PARAMS))
for pol in POLICY_NAMES:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    fig.suptitle(f'Morris elementary effects — {pol}')
    for ax, obj in zip(axes.flat, OBJECTIVES):
        Si = morris_results[pol][obj]
        ax.bar(x - 0.2, Si['mu_star'], 0.35, label='μ*',  color=palette[pol])
        ax.bar(x + 0.2, Si['sigma'],   0.35, label='σ',   color='gray', alpha=0.6)
        ax.set_xticks(x); ax.set_xticklabels(MORRIS_PARAMS)
        ax.set_title(obj.replace('years_above_temperature_threshold', 'yrs > 2°C'))
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## Reflection Questions

**1. Why is Morris used instead of Sobol for the normative parameter analysis?**


*(Write your answer here.)*


**2. Physical vs. normative sensitivity.** What do ET and Morris together reveal about
which type of uncertainty dominates each outcome?


*(Write your answer here.)*


**3. Policy-conditional sensitivity.** Which normative parameter's μ\* changed most
between the two policies? What does this imply for decision-making?


*(Write your answer here.)*
